# Residual RNN Ablation Training for STM32 Deployment

This notebook trains the same compact residual RNN forecast pipeline as the int8 residual notebook, but uses only the original sensor columns from the dataset. It avoids engineered time, delta, and rolling-window features so you can compare the raw-input ablation directly against the engineered-feature model.


In [1]:
from pathlib import Path
import os
import random

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

Matplotlib is building the font cache; this may take a moment.


In [2]:
SEQ_LENGTH = 48          # 24 hours of 30-minute samples
HORIZON = 24             # next 12 hours at 30-minute resolution
RESAMPLE_RULE = '30min'
BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE

**Explanation:**
`SEQ_LENGTH = 48`:
- Each input example contains 48 time steps 
- Since data is resampled to 30-minute intervals, that corresponds to 48 * 30 minutes = 1,440 minutes = 24 hours

`HORIZON = 24`:
- The model predicts the next 24 time steps.
- Since data is resampled to 30-minute interval, that corresponds to 24 * 30 minutes = 720 minutes = 12 hours

`RESAMPLE_RULE = '30min'`:
- Tells Pandas to aggregate the raw sensor readings into one row every 30 minutes.
- Basically sensors capture data at slightly different times, so we aggregate them within a 30 minute window.

`BATCH_SIZE = 64`:
- When training the model, 64 examples are processed before updating the model weights.

`AUTOTUNE = tf.data.autotune`:
- Tells TensorFlow to automatically decide how aggressively it should prepare the next batches of data in the background while the model is training.

In [3]:
EXCLUDED_FILES = {
    'measurements_2000-01-01.csv',
    'power_measurements.csv',
}

ARTIFACT_DIR = Path('machine_learning/rnn/artifacts')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_TFLITE_PATH = Path('machine_learning/rnn/ablation_rnn_int8.tflite')

repo_root_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
REPO_ROOT = next((path for path in repo_root_candidates if (path / 'measurements').is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError(f'Could not locate repo root. Checked: {repo_root_candidates}')

DATA_DIR = REPO_ROOT / 'measurements'
print('Repo root:', REPO_ROOT.resolve())
print('Artifact dir:', ARTIFACT_DIR.resolve())
print('Tracked TFLite export path:', EXPORT_TFLITE_PATH.resolve())

Repo root: /Users/tevinachong/Documents/software_projects/TinyML-Home-Weather-Forecasting
Artifact dir: /Users/tevinachong/Documents/software_projects/TinyML-Home-Weather-Forecasting/machine_learning/rnn/machine_learning/rnn/artifacts
Tracked TFLite export path: /Users/tevinachong/Documents/software_projects/TinyML-Home-Weather-Forecasting/machine_learning/rnn/machine_learning/rnn/ablation_rnn_int8.tflite


In [4]:
csv_files = sorted(
    path for path in DATA_DIR.glob('*measurements*.csv')
    if path.name not in EXCLUDED_FILES
)

if not csv_files:
    raise FileNotFoundError(f'No measurement CSV files found in {DATA_DIR}')

frames = []
skipped_files = []
for csv_file in csv_files:
    if csv_file.stat().st_size == 0:
        skipped_files.append((csv_file.name, 'empty file'))
        continue
    try:
        df_temp = pd.read_csv(csv_file)
    except pd.errors.EmptyDataError:
        skipped_files.append((csv_file.name, 'empty data'))
        continue
    frames.append(df_temp)

if not frames:
    raise ValueError('No readable measurement CSVs were loaded.')

raw_df = pd.concat(frames, ignore_index=True)
raw_df = raw_df[raw_df['quantity'].isin(['temperature_c', 'humidity_pct', 'pressure_pa', 'lux_lx', 'is_raining'])].copy()
raw_df['timestamp_iso8601'] = pd.to_datetime(raw_df['timestamp_iso8601'], utc=True, errors='coerce')
raw_df['value'] = pd.to_numeric(raw_df['value'], errors='coerce')
raw_df = raw_df.dropna(subset=['timestamp_iso8601', 'value'])

raw_df.head(10)

,timestamp_iso8601,sensor,quantity,value,unit
0,2025-09-04 22:13:36+00:00,veml7700,lux_lx,26.342400,lx
1,2025-09-04 22:13:37+00:00,bme280,temperature_c,29.042791,degC
2,2025-09-04 22:13:37+00:00,bme280,pressure_pa,100963.773438,Pa
3,2025-09-04 22:13:37+00:00,bme280,humidity_pct,64.752380,pct
4,2025-09-04 22:13:43+00:00,lm393,is_raining,0.000000,NaN
5,2025-09-04 22:13:43+00:00,veml7700,lux_lx,27.417599,lx
6,2025-09-04 22:13:43+00:00,bme280,temperature_c,29.022619,degC
7,2025-09-04 22:13:43+00:00,bme280,pressure_pa,100968.562500,Pa
8,2025-09-04 22:13:43+00:00,bme280,humidity_pct,64.420753,pct
9,2025-09-04 22:13:52+00:00,lm393,is_raining,0.000000,NaN


In [5]:
wide_df = (
    raw_df
    .pivot_table(index='timestamp_iso8601', columns='quantity', values='value', aggfunc='mean')
    .sort_index()
)

resampled_df = wide_df.resample(RESAMPLE_RULE).mean()
resampled_df['is_raining'] = resampled_df['is_raining'].fillna(0.0).clip(0.0, 1.0)
numeric_cols = [col for col in resampled_df.columns if col != 'is_raining']
resampled_df[numeric_cols] = resampled_df[numeric_cols].interpolate(method='time', limit_direction='both')
resampled_df = resampled_df.dropna().copy()

print('Loaded files:', len(frames))
print('Skipped files:', skipped_files)
print('Raw rows:', len(raw_df))
print('Resampled rows:', len(resampled_df))
resampled_df.head()

Loaded files: 116
Skipped files: []
Raw rows: 3710079
Resampled rows: 9603


quantity,humidity_pct,is_raining,lux_lx,pressure_pa,temperature_c
timestamp_iso8601,,,,,
2025-09-04 22:00:00+00:00,63.341935,0.0,26.594607,100973.617845,28.172483
2025-09-04 22:30:00+00:00,69.947520,0.0,42.181484,100969.579486,28.347315
2025-09-04 23:00:00+00:00,75.398518,0.0,97.414887,100973.551907,30.579943
2025-09-04 23:30:00+00:00,76.887413,0.0,97.793978,101008.485453,30.319356
2025-09-05 00:00:00+00:00,79.289795,0.0,97.822113,101026.242403,29.734615


In [6]:
feature_df = resampled_df.copy()

feature_cols = [
    'temperature_c',
    'humidity_pct',
    'pressure_pa',
    'lux_lx',
    'is_raining',
]

target_col = 'temperature_c'
temp_feature_index = feature_cols.index('temperature_c')
feature_df[feature_cols].head()


quantity,temperature_c,humidity_pct,pressure_pa,lux_lx,is_raining
timestamp_iso8601,,,,,
2025-09-04 22:00:00+00:00,28.172483,63.341935,100973.617845,26.594607,0.0
2025-09-04 22:30:00+00:00,28.347315,69.947520,100969.579486,42.181484,0.0
2025-09-04 23:00:00+00:00,30.579943,75.398518,100973.551907,97.414887,0.0
2025-09-04 23:30:00+00:00,30.319356,76.887413,101008.485453,97.793978,0.0
2025-09-05 00:00:00+00:00,29.734615,79.289795,101026.242403,97.822113,0.0


In [7]:
n_rows = len(feature_df)
train_row_end = int(n_rows * 0.70)
valid_row_end = int(n_rows * 0.85)

train_rows = feature_df.iloc[:train_row_end].copy()
valid_rows = feature_df.iloc[train_row_end:valid_row_end].copy()
test_rows = feature_df.iloc[valid_row_end:].copy()

scale_cols = [
    'humidity_pct',
    'pressure_pa',
    'lux_lx',
]

scaler = StandardScaler()
scaler.fit(train_rows[scale_cols])

scaled_feature_df = feature_df.copy()
scaled_feature_df.loc[:, scale_cols] = scaler.transform(feature_df[scale_cols])

train_cut_timestamp = feature_df.index[train_row_end]
valid_cut_timestamp = feature_df.index[valid_row_end]

print('Train rows:', len(train_rows))
print('Valid rows:', len(valid_rows))
print('Test rows:', len(test_rows))
print('Train cut:', train_cut_timestamp)
print('Valid cut:', valid_cut_timestamp)


Train rows: 6722
Valid rows: 1440
Test rows: 1441
Train cut: 2026-01-22 23:00:00+00:00
Valid cut: 2026-02-21 23:00:00+00:00


In [8]:
def build_supervised_windows(df, feature_columns, target_column, seq_length, horizon):
    feature_matrix = df[feature_columns].to_numpy(dtype=np.float32)
    target_vector = df[target_column].to_numpy(dtype=np.float32)
    timestamps = df.index.to_numpy()

    x_list = []
    y_list = []
    baseline_list = []
    target_timestamps = []

    max_start = len(df) - seq_length - horizon + 1
    for start in range(max_start):
        stop = start + seq_length
        horizon_stop = stop + horizon

        input_window = feature_matrix[start:stop]
        future_target = target_vector[stop:horizon_stop]

        # For each future step, use the value from the same half-hour one day earlier.
        previous_day_baseline = input_window[:horizon, temp_feature_index]

        x_list.append(input_window)
        y_list.append(future_target)
        baseline_list.append(previous_day_baseline)
        target_timestamps.append(timestamps[stop])

    return (
        np.asarray(x_list, dtype=np.float32),
        np.asarray(y_list, dtype=np.float32),
        np.asarray(baseline_list, dtype=np.float32),
        np.asarray(target_timestamps),
    )


X_all, y_all, baseline_all, sample_timestamps = build_supervised_windows(
    scaled_feature_df,
    feature_cols,
    target_col,
    SEQ_LENGTH,
    HORIZON,
)

sample_timestamps = pd.to_datetime(sample_timestamps, utc=True)
train_cut_ts = pd.Timestamp(train_cut_timestamp)
valid_cut_ts = pd.Timestamp(valid_cut_timestamp)
if train_cut_ts.tzinfo is None:
    train_cut_ts = train_cut_ts.tz_localize('UTC')
else:
    train_cut_ts = train_cut_ts.tz_convert('UTC')
if valid_cut_ts.tzinfo is None:
    valid_cut_ts = valid_cut_ts.tz_localize('UTC')
else:
    valid_cut_ts = valid_cut_ts.tz_convert('UTC')

train_mask = sample_timestamps < train_cut_ts
valid_mask = (sample_timestamps >= train_cut_ts) & (sample_timestamps < valid_cut_ts)
test_mask = sample_timestamps >= valid_cut_ts

X_train, y_train, baseline_train = X_all[train_mask], y_all[train_mask], baseline_all[train_mask]
X_valid, y_valid, baseline_valid = X_all[valid_mask], y_all[valid_mask], baseline_all[valid_mask]
X_test, y_test, baseline_test = X_all[test_mask], y_all[test_mask], baseline_all[test_mask]

print('Train windows:', X_train.shape, y_train.shape)
print('Valid windows:', X_valid.shape, y_valid.shape)
print('Test windows:', X_test.shape, y_test.shape)

baseline_valid_mae = float(np.mean(np.abs(y_valid - baseline_valid)))
baseline_test_mae = float(np.mean(np.abs(y_test - baseline_test)))
print(f'Naive day-ago baseline valid MAE: {baseline_valid_mae:.4f} C')
print(f'Naive day-ago baseline test MAE:  {baseline_test_mae:.4f} C')

Train windows: (6674, 48, 5) (6674, 24)
Valid windows: (1440, 48, 5) (1440, 24)
Test windows: (1418, 48, 5) (1418, 24)
Naive day-ago baseline valid MAE: 0.0543 C
Naive day-ago baseline test MAE:  0.9971 C


In [9]:
def make_dataset(X, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if training:
        ds = ds.shuffle(min(len(X), 8192), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds


train_ds = make_dataset(X_train, y_train, training=True)
valid_ds = make_dataset(X_valid, y_valid, training=False)
test_ds = make_dataset(X_test, y_test, training=False)

horizon_weights = tf.constant(np.linspace(1.0, 1.35, HORIZON), dtype=tf.float32)

def weighted_huber(y_true, y_pred):
    error = y_true - y_pred
    abs_error = tf.abs(error)
    delta = 1.5
    quadratic = tf.minimum(abs_error, delta)
    linear = abs_error - quadratic
    huber = 0.5 * tf.square(quadratic) + delta * linear
    weighted = huber * horizon_weights
    return tf.reduce_mean(weighted, axis=-1)


def build_residual_rnn(config):
    inputs = tf.keras.Input(shape=(SEQ_LENGTH, len(feature_cols)), name='sensor_history')
    baseline = tf.keras.layers.Lambda(
        lambda x: x[:, :HORIZON, temp_feature_index],
        name='previous_day_baseline',
    )(inputs)

    x = tf.keras.layers.SimpleRNN(
        config['rnn_units_1'],
        return_sequences=True,
        unroll=True,
        activation='tanh',
        name='rnn_1',
    )(inputs)
    x = tf.keras.layers.Dropout(config['dropout'], name='dropout_1')(x)
    x = tf.keras.layers.SimpleRNN(
        config['rnn_units_2'],
        unroll=True,
        activation='tanh',
        name='rnn_2',
    )(x)
    x = tf.keras.layers.Dense(config['dense_units'], activation='relu', name='dense_1')(x)
    residual = tf.keras.layers.Dense(HORIZON, name='residual_delta_c')(x)
    outputs = tf.keras.layers.Add(name='forecast_c')([baseline, residual])

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name='ablation_rnn_forecaster')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=config['learning_rate']),
        loss=weighted_huber,
        metrics=[tf.keras.metrics.MeanAbsoluteError(name='mae')],
    )
    return model


search_configs = [
    {'name': 'cfg_small',  'rnn_units_1': 24, 'rnn_units_2': 16, 'dense_units': 24, 'dropout': 0.05, 'learning_rate': 2e-3},
    {'name': 'cfg_mid',    'rnn_units_1': 32, 'rnn_units_2': 24, 'dense_units': 32, 'dropout': 0.10, 'learning_rate': 1e-3},
    {'name': 'cfg_wide',   'rnn_units_1': 40, 'rnn_units_2': 24, 'dense_units': 32, 'dropout': 0.10, 'learning_rate': 8e-4},
    {'name': 'cfg_deep',   'rnn_units_1': 32, 'rnn_units_2': 32, 'dense_units': 40, 'dropout': 0.15, 'learning_rate': 8e-4},
    {'name': 'cfg_edge',   'rnn_units_1': 48, 'rnn_units_2': 24, 'dense_units': 32, 'dropout': 0.05, 'learning_rate': 7e-4},
]

def count_params_for_config(config):
    return build_residual_rnn(config).count_params()

[(cfg['name'], count_params_for_config(cfg)) for cfg in search_configs]

[('cfg_small', 2384),
 ('cfg_mid', 4176),
 ('cfg_wide', 4992),
 ('cfg_deep', 5600),
 ('cfg_edge', 5936)]

In [10]:
training_results = []
best_model = None
best_history = None
best_config = None
best_valid_mae = np.inf

for config in search_configs:
    print(f"\n=== Training {config['name']} ===")
    model = build_residual_rnn(config)
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='val_mae', patience=8, mode='min', restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_mae', factor=0.5, patience=3, min_lr=1e-5, mode='min'),
    ]

    history = model.fit(
        train_ds,
        validation_data=valid_ds,
        epochs=50,
        verbose=1,
        callbacks=callbacks,
    )

    valid_metrics = model.evaluate(valid_ds, verbose=0, return_dict=True)
    params = model.count_params()
    training_results.append({
        'name': config['name'],
        'params': params,
        'val_loss': float(valid_metrics['loss']),
        'val_mae': float(valid_metrics['mae']),
    })
    print(f"Validation MAE for {config['name']}: {valid_metrics['mae']:.4f} C | params={params:,}")

    if valid_metrics['mae'] < best_valid_mae:
        best_valid_mae = float(valid_metrics['mae'])
        best_model = model
        best_history = history.history
        best_config = config

results_df = pd.DataFrame(training_results).sort_values(['val_mae', 'params']).reset_index(drop=True)
print('\nSearch results:')
display(results_df)
print('Best config:', best_config)


=== Training cfg_small ===
Epoch 1/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 0.9352 - mae: 0.9218 - val_loss: 0.0064 - val_mae: 0.1027 - learning_rate: 0.0020
Epoch 2/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.9268 - mae: 0.9078 - val_loss: 0.0090 - val_mae: 0.1224 - learning_rate: 0.0020
Epoch 3/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.9259 - mae: 0.9069 - val_loss: 0.0073 - val_mae: 0.0984 - learning_rate: 0.0020
Epoch 4/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.9248 - mae: 0.9056 - val_loss: 0.0077 - val_mae: 0.1071 - learning_rate: 0.0020
Epoch 5/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.9241 - mae: 0.9043 - val_loss: 0.0040 - val_mae: 0.0716 - learning_rate: 0.0020
Epoch 6/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.9233 - mae: 0.9057 - val_loss: 0.0023 - val_mae: 0.0497 - learning_rate: 0.0020
Epoch 7/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.9225 - mae: 0.9082 - val_loss: 0.0099 - val_mae: 0.1244 - 

,name,params,val_loss,val_mae
0,cfg_small,2384,0.002303,0.049678
1,cfg_deep,5600,0.002614,0.063798
2,cfg_edge,5936,0.002723,0.065419
3,cfg_wide,4992,0.003180,0.066060
4,cfg_mid,4176,0.003087,0.069589


Best config: {'name': 'cfg_small', 'rnn_units_1': 24, 'rnn_units_2': 16, 'dense_units': 24, 'dropout': 0.05, 'learning_rate': 0.002}


In [11]:
final_metrics = {
    'baseline_valid_mae_c': baseline_valid_mae,
    'baseline_test_mae_c': baseline_test_mae,
}

valid_eval = best_model.evaluate(valid_ds, verbose=0, return_dict=True)
test_eval = best_model.evaluate(test_ds, verbose=0, return_dict=True)
final_metrics['model_valid_mae_c'] = float(valid_eval['mae'])
final_metrics['model_test_mae_c'] = float(test_eval['mae'])
final_metrics['test_mae_gain_c'] = baseline_test_mae - float(test_eval['mae'])
final_metrics['test_skill_vs_baseline_pct'] = 100.0 * final_metrics['test_mae_gain_c'] / max(baseline_test_mae, 1e-6)

y_valid_pred = best_model.predict(X_valid, batch_size=BATCH_SIZE, verbose=0)
y_test_pred = best_model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)

per_horizon_test_mae = np.mean(np.abs(y_test - y_test_pred), axis=0)
per_horizon_baseline_mae = np.mean(np.abs(y_test - baseline_test), axis=0)

metrics_df = pd.DataFrame([final_metrics])
display(metrics_df)

comparison_df = pd.DataFrame({
    'step_ahead': np.arange(1, HORIZON + 1),
    'baseline_mae_c': per_horizon_baseline_mae,
    'model_mae_c': per_horizon_test_mae,
    'gain_c': per_horizon_baseline_mae - per_horizon_test_mae,
})
display(comparison_df)

best_model_path = ARTIFACT_DIR / 'best_ablation_rnn.keras'
best_model.save(best_model_path)
print('Saved best Keras model to:', best_model_path.resolve())

if final_metrics['model_test_mae_c'] >= baseline_test_mae:
    print('Model is not yet beating the day-ago baseline on test. Expand the search grid or train longer before deployment.')
else:
    print('Model beats the day-ago baseline on test and is ready for quantization.')

,baseline_valid_mae_c,baseline_test_mae_c,model_valid_mae_c,model_test_mae_c,test_mae_gain_c,test_skill_vs_baseline_pct
0,0.054303,0.99711,0.049678,0.988973,0.008136,0.81598


,step_ahead,baseline_mae_c,model_mae_c,gain_c
0,1,0.989851,0.956469,0.033382
1,2,0.990008,0.959941,0.030068
2,3,0.990404,0.958401,0.032003
3,4,0.991291,0.972840,0.018451
4,5,0.991907,0.961322,0.030585
5,6,0.992235,0.961010,0.031225
6,7,0.992431,0.968234,0.024197
7,8,0.992546,0.970253,0.022293
8,9,0.992569,0.989552,0.003017
9,10,0.992800,0.976462,0.016338


Saved best Keras model to: /Users/tevinachong/Documents/software_projects/TinyML-Home-Weather-Forecasting/machine_learning/rnn/machine_learning/rnn/artifacts/best_ablation_rnn.keras
Model beats the day-ago baseline on test and is ready for quantization.


In [12]:
best_model.summary()

def estimate_tflite_sram_kb(tflite_bytes):
    interpreter = tf.lite.Interpreter(model_content=tflite_bytes)
    interpreter.allocate_tensors()
    tensor_details = interpreter.get_tensor_details()

    def tensor_ram_bytes(detail):
        dtype = np.dtype(detail['dtype'])
        shape = detail.get('shape_signature', detail.get('shape'))
        shape = [1 if dim < 0 else dim for dim in shape]
        return int(np.prod(shape)) * dtype.itemsize

    sram_bytes = sum(
        tensor_ram_bytes(detail)
        for detail in tensor_details
        if detail.get('buffer', 0) == 0 or detail.get('is_variable', False)
    )
    return sram_bytes / 1024.0

def representative_dataset():
    for batch_x, _ in train_ds.unbatch().batch(1).take(400):
        yield [tf.cast(batch_x, tf.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_int8_bytes = converter.convert()
tflite_path = ARTIFACT_DIR / 'ablation_rnn_int8.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_int8_bytes)
with open(EXPORT_TFLITE_PATH, 'wb') as f:
    f.write(tflite_int8_bytes)

size_bytes = tflite_path.stat().st_size
size_kb = size_bytes / 1024.0
estimated_sram_kb = estimate_tflite_sram_kb(tflite_int8_bytes)

print(f'TFLite int8 artifact path: {tflite_path.resolve()}')
print(f'TFLite int8 tracked path: {EXPORT_TFLITE_PATH.resolve()}')
print(f'TFLite int8 size: {size_bytes} bytes ({size_kb:.2f} KB)')
print(f'Estimated tensor SRAM: {estimated_sram_kb:.2f} KB')

Model: "ablation_rnn_forecaster"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ sensor_history      │ (None, 48, 5)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rnn_1 (SimpleRNN)   │ (None, 48, 24)    │        720 │ sensor_history[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 48, 24)    │          0 │ rnn_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rnn_2 (SimpleRNN)   │ (None, 16)        │        656 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 24)        │        408 │ rnn_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ previous_day_basel… │ (None, 24)        │          0 │ sensor_history[0… │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ residual_delta_c    │ (None, 24)        │        600 │ dense_1[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ forecast_c (Add)    │ (None, 24)        │          0 │ previous_day_bas… │
│                     │                   │            │ residual_delta_c… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 7,154 (27.95 KB)

 Trainable params: 2,384 (9.31 KB)

 Non-trainable params: 0 (0.00 B)

...

INFO:tensorflow:Assets written to: /var/folders/_8/4njq_61d7lg7gd61fqck34dr0000gn/T/tmp36ym0gh9/assets


Saved artifact at '/var/folders/_8/4njq_61d7lg7gd61fqck34dr0000gn/T/tmp36ym0gh9'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 48, 5), dtype=tf.float32, name='sensor_history')
Output Type:
  TensorSpec(shape=(None, 24), dtype=tf.float32, name=None)
Captures:
  5112881120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5112787920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5112881648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5112788800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5112781056: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5112788624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5112781936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5112783344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5112779296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  5112775952: TensorSpec(shape=(), dtype=tf.resource, name=None)


/Users/tevinachong/Documents/software_projects/TinyML-Home-Weather-Forecasting/.venv/lib/python3.10/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1775060222.012949 1055369 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1775060222.012958 1055369 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1775060222.013176 1055369 reader.cc:83] Reading SavedModel from: /var/folders/_8/4njq_61d7lg7gd61fqck34dr0000gn/T/tmp36ym0gh9
I0000 00:00:1775060222.014168 1055369 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1775060222.014172 1055369 reader.cc:147] Reading SavedModel debug info (if present) from: /var/folders/_8/4njq_61d7lg7gd61fqck34dr0000gn/T/tmp36ym0gh9
I0000 00:00:1775060222.024132 1055369 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1775060222.0

TFLite int8 artifact path: /Users/tevinachong/Documents/software_projects/TinyML-Home-Weather-Forecasting/machine_learning/rnn/machine_learning/rnn/artifacts/ablation_rnn_int8.tflite
TFLite int8 tracked path: /Users/tevinachong/Documents/software_projects/TinyML-Home-Weather-Forecasting/machine_learning/rnn/machine_learning/rnn/ablation_rnn_int8.tflite
TFLite int8 size: 354512 bytes (346.20 KB)
Estimated tensor SRAM: 22.88 KB


/Users/tevinachong/Documents/software_projects/TinyML-Home-Weather-Forecasting/.venv/lib/python3.10/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [13]:
interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

print('Input details:', input_details)
print('Output details:', output_details)

input_scale, input_zero_point = input_details['quantization']
output_scale, output_zero_point = output_details['quantization']

def quantize_input(x):
    return np.clip(np.round(x / input_scale + input_zero_point), -128, 127).astype(np.int8)

def dequantize_output(x):
    return (x.astype(np.float32) - output_zero_point) * output_scale

tflite_preds = []
for i in range(len(X_test)):
    sample_x = X_test[i:i+1].astype(np.float32)
    interpreter.set_tensor(input_details['index'], quantize_input(sample_x))
    interpreter.invoke()
    raw_output = interpreter.get_tensor(output_details['index'])
    tflite_preds.append(dequantize_output(raw_output)[0])

tflite_preds = np.asarray(tflite_preds, dtype=np.float32)
tflite_test_mae = float(np.mean(np.abs(y_test - tflite_preds)))
print(f'TFLite int8 test MAE: {tflite_test_mae:.4f} C')
print(f'Test baseline MAE:   {baseline_test_mae:.4f} C')
print(f'TFLite gain vs baseline: {baseline_test_mae - tflite_test_mae:.4f} C')

preview_index = 0
preview_df = pd.DataFrame({
    'actual_c': y_test[preview_index],
    'baseline_c': baseline_test[preview_index],
    'keras_pred_c': y_test_pred[preview_index],
    'tflite_pred_c': tflite_preds[preview_index],
})

preview_df["baseline_diff"] = np.abs(preview_df["baseline_c"] - preview_df["actual_c"])
preview_df["keras_diff"] = np.abs(preview_df["keras_pred_c"] - preview_df["actual_c"])
preview_df["tflite_diff"] = np.abs(preview_df["tflite_pred_c"] - preview_df["actual_c"])
display(preview_df)


Input details: {'name': 'serving_default_sensor_history:0', 'index': 0, 'shape': array([ 1, 48,  5], dtype=int32), 'shape_signature': array([-1, 48,  5], dtype=int32), 'dtype': <class 'numpy.int8'>, 'quantization': (0.18376392126083374, -110), 'quantization_parameters': {'scales': array([0.18376392], dtype=float32), 'zero_points': array([-110], dtype=int32), 'quantized_dimension': 0, 'block_size': 0}, 'sparsity_parameters': {}}
Output details: {'name': 'StatefulPartitionedCall_1:0', 'index': 657, 'shape': array([ 1, 24], dtype=int32), 'shape_signature': array([-1, 24], dtype=int32), 'dtype': <class 'numpy.int8'>, 'quantization': (0.17114542424678802, -128), 'quantization_parameters': {'scales': array([0.17114542], dtype=float32), 'zero_points': array([-128], dtype=int32), 'quantized_dimension': 0, 'block_size': 0}, 'sparsity_parameters': {}}
TFLite int8 test MAE: 1.0030 C
Test baseline MAE:   0.9971 C
TFLite gain vs baseline: -0.0059 C


,actual_c,baseline_c,keras_pred_c,tflite_pred_c,baseline_diff,keras_diff,tflite_diff
0,28.525335,28.471033,28.602253,28.581285,0.054302,0.076918,0.055950
1,28.526466,28.472164,28.599707,28.581285,0.054302,0.073240,0.054819
2,28.527599,28.473295,28.588533,28.581285,0.054304,0.060934,0.053686
3,28.528730,28.474426,28.612749,28.581285,0.054304,0.084019,0.052555
4,28.529861,28.475559,28.563002,28.581285,0.054302,0.033140,0.051424
5,28.530993,28.476690,28.547232,28.581285,0.054302,0.016239,0.050293
6,28.532124,28.477821,28.557318,28.581285,0.054302,0.025194,0.049162
7,28.533255,28.478952,28.565916,28.581285,0.054302,0.032661,0.048031
8,28.534386,28.480083,28.559095,28.581285,0.054302,0.024710,0.046900
9,28.535517,28.481215,28.557667,28.581285,0.054302,0.022150,0.045769


In [14]:
print(f"Baseline MAE: {preview_df['baseline_diff'].mean():.4f} C")
print(f"Full Model MAE: {preview_df['keras_diff'].mean():.4f} C")
print(f"TFLite MAE: {preview_df['tflite_diff'].mean():.4f} C")

Baseline MAE: 0.0543 C
Full Model MAE: 0.0545 C
TFLite MAE: 0.0850 C
